In [0]:
%run ./00_config

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_assets (
  asset_id STRING, ticker STRING, asset_name STRING, asset_class STRING,
  _source_file STRING, _ingested_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver_clients (
  client_id STRING, client_name STRING, tier STRING, risk_profile STRING, updated_at TIMESTAMP,
  _source_file STRING, _ingested_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver_prices (
  asset_id STRING, price_date DATE,
  open DECIMAL(18,4), high DECIMAL(18,4), low DECIMAL(18,4), close DECIMAL(18,4), adjusted_close DECIMAL(18,4),
  volume BIGINT, updated_at TIMESTAMP,
  _source_file STRING, _ingested_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver_trades (
  trade_id STRING, client_id STRING, asset_id STRING, trade_date DATE, side STRING,
  quantity DECIMAL(18,4), unit_price DECIMAL(18,4), updated_at TIMESTAMP,
  _source_file STRING, _ingested_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver_ticks (
  asset_id STRING, tick_ts TIMESTAMP, price DECIMAL(18,4), volume BIGINT,
  _source_file STRING, _ingested_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver_quarantine (
  feed STRING, reason STRING, record STRING, _source_file STRING, quarantined_at TIMESTAMP
);

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

META = ["_raw", "_source_file", "_ingested_at", "_rescued_data"]

def tc(col, dtype):
    """try_cast: returns null instead of failing when a value can't be converted."""
    return F.expr(f"try_cast({col} AS {dtype})")

def run_silver(feed, keys, order_col, prepare):
    target = f"silver_{feed}"

    def process(batch_df, batch_id):
        session = batch_df.sparkSession

        # 1. keep the original raw values, for quarantine
        raw_cols = [c for c in batch_df.columns if not c.startswith("_")]
        batch_df = batch_df.withColumn("_raw", F.to_json(F.struct(*raw_cols)))

        # 2. cast types and get this feed's validation rules
        typed, rules = prepare(batch_df, session)
        rules["unexpected columns"] = F.col("_rescued_data").isNull()

        # 3. list every rule each row fails ("" = passed all)
        failed = [F.when(~F.coalesce(cond, F.lit(False)), F.lit(name)) for name, cond in rules.items()]
        flagged = typed.withColumn("_reason", F.concat_ws(", ", *failed))

        # 4. bad rows → quarantine
        (flagged.filter("_reason != ''")
            .select(F.lit(feed).alias("feed"), F.col("_reason").alias("reason"),
                    F.col("_raw").alias("record"), "_source_file",
                    F.current_timestamp().alias("quarantined_at"))
            .write.mode("append").saveAsTable("silver_quarantine"))

        # 5. good rows: keep only the target's columns, latest version per key
        good = flagged.filter("_reason = ''").select(*session.table(target).columns)
        w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc(), F.col("_ingested_at").desc())
        latest = good.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

        # 6. upsert
        match = " AND ".join(f"t.{k} = s.{k}" for k in keys)
        (DeltaTable.forName(session, target).alias("t")
            .merge(latest.alias("s"), match)
            .whenMatchedUpdateAll(condition=f"s.{order_col} > t.{order_col}")
            .whenNotMatchedInsertAll()
            .execute())

    before   = spark.table(target).count()
    q_before = spark.table("silver_quarantine").where(f"feed = '{feed}'").count()

    query = (spark.readStream.table(f"bronze_{feed}")
               .writeStream
               .foreachBatch(process)
               .option("checkpointLocation", f"{CHK}/{target}")
               .trigger(availableNow=True)
               .start())
    query.awaitTermination()

    after   = spark.table(target).count()
    q_after = spark.table("silver_quarantine").where(f"feed = '{feed}'").count()
    print(f"{target}: +{after - before} rows ({after} total), {q_after - q_before} quarantined")

In [0]:
def prep_assets(df, session):
    typed = df.select(F.trim("asset_id").alias("asset_id"), F.trim("ticker").alias("ticker"),
                      "asset_name", "asset_class", *META)
    rules = {"missing asset_id": F.col("asset_id").isNotNull(),
             "missing ticker":   F.col("ticker").isNotNull()}
    return typed, rules

def prep_clients(df, session):
    typed = df.select(F.trim("client_id").alias("client_id"), "client_name", "tier", "risk_profile",
                      tc("updated_at", "TIMESTAMP").alias("updated_at"), *META)
    rules = {"missing client_id":  F.col("client_id").isNotNull(),
             "missing updated_at": F.col("updated_at").isNotNull()}
    return typed, rules

def prep_prices(df, session):
    known = session.table("silver_assets").select("asset_id").withColumn("_asset_ok", F.lit(True))
    typed = (df.select(
                F.trim("asset_id").alias("asset_id"),
                tc("price_date", "DATE").alias("price_date"),
                *[tc(c, "DECIMAL(18,4)").alias(c) for c in ["open", "high", "low", "close", "adjusted_close"]],
                F.expr("CAST(try_cast(volume AS DOUBLE) AS BIGINT)").alias("volume"),
                tc("updated_at", "TIMESTAMP").alias("updated_at"), *META)
             .join(known, "asset_id", "left"))
    rules = {"unknown asset":      F.col("_asset_ok"),
             "bad price_date":     F.col("price_date").isNotNull(),
             "bad close":          F.col("close") > 0,
             "high below low":     F.col("high") >= F.col("low"),
             "missing updated_at": F.col("updated_at").isNotNull()}
    return typed, rules

def prep_trades(df, session):
    assets  = session.table("silver_assets").select("asset_id").withColumn("_asset_ok", F.lit(True))
    clients = session.table("silver_clients").select("client_id").withColumn("_client_ok", F.lit(True))
    typed = (df.select(
                F.trim("trade_id").alias("trade_id"), F.trim("client_id").alias("client_id"),
                F.trim("asset_id").alias("asset_id"),
                tc("trade_date", "DATE").alias("trade_date"),
                F.upper(F.trim("side")).alias("side"),
                tc("quantity", "DECIMAL(18,4)").alias("quantity"),
                tc("unit_price", "DECIMAL(18,4)").alias("unit_price"),
                tc("updated_at", "TIMESTAMP").alias("updated_at"), *META)
             .join(assets, "asset_id", "left")
             .join(clients, "client_id", "left"))
    rules = {"missing trade_id":   F.col("trade_id").isNotNull(),
             "unknown asset":      F.col("_asset_ok"),
             "unknown client":     F.col("_client_ok"),
             "bad trade_date":     F.col("trade_date").isNotNull(),
             "bad side":           F.col("side").isin("BUY", "SELL"),
             "bad quantity":       F.col("quantity") > 0,
             "bad unit_price":     F.col("unit_price") > 0,
             "missing updated_at": F.col("updated_at").isNotNull()}
    return typed, rules

def prep_ticks(df, session):
    known = session.table("silver_assets").select("asset_id").withColumn("_asset_ok", F.lit(True))
    typed = (df.select(
                F.trim("asset_id").alias("asset_id"),
                tc("created_at", "TIMESTAMP").alias("tick_ts"),
                tc("close_price", "DECIMAL(18,4)").alias("price"),
                F.expr("CAST(try_cast(volume AS DOUBLE) AS BIGINT)").alias("volume"), *META)
             .join(known, "asset_id", "left"))
    rules = {"unknown asset": F.col("_asset_ok"),
             "bad tick_ts":   F.col("tick_ts").isNotNull(),
             "bad price":     F.col("price") > 0}
    return typed, rules

In [0]:
display(spark.table("silver_ticks").limit(20))   # what it produced

In [0]:
run_silver("assets",  ["asset_id"],               "_ingested_at", prep_assets)
run_silver("clients", ["client_id"],              "updated_at",   prep_clients)
run_silver("prices",  ["asset_id", "price_date"], "updated_at",   prep_prices)
run_silver("trades",  ["trade_id"],               "updated_at",   prep_trades)
run_silver("ticks",   ["asset_id", "tick_ts"],    "_ingested_at", prep_ticks)

In [0]:
display(spark.table("silver_ticks").limit(20))   # what it produced